# Keep-alive and resource monitor

Run the one code cell and leave the notebook open while long batches run. It prints CPU cores in use, RAM used against the pod's limit, free disk and the number of pipeline processes once a minute, which keeps the kernel active so the pod is not shut down for inactivity. It reads system files only and holds almost no memory.

In [ ]:
# Keep-alive monitor. Why (user, 7 Oct 2026): the pod shuts down after a while without kernel activity, which stopped
# long batch runs twice. This loop keeps the kernel busy and prints the pod's load once a minute. It is tiny (no extra
# packages, reads /proc and the cgroup files only), so it cannot cause an out-of-memory stop itself.
# Stop it with the notebook's stop button (Kernel > Interrupt).
import os, time, subprocess
from datetime import datetime

def _read(p, default=None):
    try:
        return open(p).read().strip()
    except OSError:
        return default

def mem_used_limit():
    used = _read("/sys/fs/cgroup/memory.current")
    limit = _read("/sys/fs/cgroup/memory.max")
    if used and limit and limit != "max":
        return int(used), int(limit)
    info = dict(l.split(":", 1) for l in open("/proc/meminfo"))
    tot = int(info["MemTotal"].split()[0]) * 1024
    avail = int(info["MemAvailable"].split()[0]) * 1024
    return tot - avail, tot

def cpu_usage_usec():
    s = _read("/sys/fs/cgroup/cpu.stat", "")
    for line in s.splitlines():
        if line.startswith("usage_usec"):
            return int(line.split()[1])
    return None

def cpu_quota():
    q = _read("/sys/fs/cgroup/cpu.max", "max 100000").split()
    return os.cpu_count() if q[0] == "max" else int(q[0]) / int(q[1])

def pipeline_jobs():
    out = subprocess.run(["pgrep", "-af", "sar_pipeline"], capture_output=True, text=True).stdout.splitlines()
    return [l for l in out if "pgrep" not in l]

prev, t_prev = cpu_usage_usec(), time.time()
while True:
    time.sleep(60)
    now, t_now = cpu_usage_usec(), time.time()
    cores = (now - prev) / 1e6 / (t_now - t_prev) if now is not None and prev is not None else float("nan")
    prev, t_prev = now, t_now
    used, limit = mem_used_limit()
    st = os.statvfs("/home/jovyan")
    free_gb = st.f_bavail * st.f_frsize / 1e9
    pct = 100 * used / limit
    warn = "  <-- RAM above 85 %" if pct > 85 else ""
    print(f"{datetime.now():%H:%M:%S}  CPU {cores:4.1f}/{cpu_quota():.0f} cores  RAM {used/1e9:5.1f}/{limit/1e9:.0f} GB "
          f"({pct:4.1f} %)  disk free {free_gb:5.0f} GB  pipeline processes {len(pipeline_jobs())}{warn}", flush=True)
